In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Identifcation of Scene Transitions in Movies Using Gemini

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/scenetransition.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fuse-cases%2Fscenetransition.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/use-cases/scenetransition.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/scenetransition.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/scenetransition.ipynb">
      <img width="32px" src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/scenetransition.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/scenetransition.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/scenetransition.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/scenetransition.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/scenetransition.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>

| Author |
| --- |
| [Vijaylaxmi Lendale](https://github.com/VJlaxmi) |

## Overview

- This notebook demonstrates how to use Gemini to automatically detect **scene transitions** in videos using both **video content** and associated **subtitle (VTT) files**.\n

## Get started

### Install Google Gen AI SDK and other required packages


In [ ]:
%pip install -q -U google-cloud-storage google-genai

### Authenticate your notebook environment (Colab only)

If you're running this notebook on Google Colab, run the cell below to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
# Use the environment variable if the user doesn't provide Project ID.
import os

PROJECT = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
LOCATION = "us-central1"
if not PROJECT or PROJECT == "[your-project-id]":
    PROJECT = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

from google import genai

client = genai.Client(vertexai=True, project=PROJECT, location=LOCATION)

### Import libraries

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime
import json
import os
import sys

import google.generativeai as genai
import vertexai
from vertexai.generative_models import GenerationConfig, GenerativeModel, Part

### output schema and model config

In [ ]:
response_schema_sequence_extraction = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "scene_number": {
                "type": "integer",
                "description": "The scene number in sequential order, e.g., 1, 2, 3.",
            },
            "start_time": {
                "type": "string",
                "description": "The start time of the scene in the format HRS:MIN:SEC (e.g., 01:15:30, 00:12:45).",
                "pattern": "^([0-9]{2}):([0-9]{2}):([0-9]{2})$",
            },
            "end_time": {
                "type": "string",
                "description": "The end time of the scene in the format HRS:MIN:SEC (e.g., 01:20:45, 00:20:18).",
                "pattern": "^([0-9]{2}):([0-9]{2}):([0-9]{2})$",
            },
            "description": {
                "type": "string",
                "description": "A brief description of the scene.",
            },
        },
        "required": ["scene_number", "start_time", "end_time", "description"],
    },
}

In [ ]:
vertexai.init(project=PROJECT, location=LOCATION)
model = GenerativeModel("gemini-2.0-pro")

In [ ]:
config = GenerationConfig(
    temperature=0,
    top_p=1,
    max_output_tokens=8000,
    response_mime_type="application/json",
    response_schema=response_schema_sequence_extraction,
    candidate_count=3,
)

In [ ]:
# Base path
base_path = "gs://<yourbucket>/<yourvideofolder>"

# One-shot paths
one_shot_video_path = "example.mp4"
one_shot_vtt_path = "example.vtt"

# Input video paths
input_video_path = f"{base_path}inputvideo.mp4"
input_vtt_path = f"{base_path}inputvtt.vtt"

# Print paths
print(f"One-shot Video Path: {one_shot_video_path}")
print(f"One-shot VTT Path: {one_shot_vtt_path}")
print(f"Input Video Path: {input_video_path}")
print(f"Input VTT Path: {input_vtt_path}")

### Prompt

In [ ]:
base_instructions = """
You are a multimodal Scene Boundary Detector.
Your task is to analyze a video along with its VTT (subtitle) file to identify cohesive and meaningful scene transitions, 
ensuring accurate segmentation into self-contained scenes with distinct narrative arcs.
** Key criteria for identifying scene transitions:** 
- Narrative Changes: Transitions must reflect a significant shift in story elements such as location, time, characters, or topic of dialogue.
- Don't treat jump-cuts or insert shots as transitions unless they signify meaningful narrative shifts.
- Visual Cues: Changes in location, character appearance, or recognition of visual elements strongly indicate scene changes.
- Dialogue Topics: Continuous dialogues between a stable set of characters typically belong to a single scene; changes in dialogue themes can signal scene transitions.
- Audio Elements: Shifts in background music or sound effects often accompany scene transitions, reinforcing narrative changes.
- Cohesion: Ensure each identified scene is cohesive, with a contained beginning, middle, and end, contributing to the overarching narrative.
Ensure that the scene transition timestamps you identify strictly fall within the start and end time boundaries of the input video, and accurately reflect the exact position of the scene boundaries in the video.
"""

In [ ]:
oneshot_parts = [
    Part.from_uri(input_video_path, "video/mp4"),
    Part.from_uri(input_vtt_path, "text/vtt"),
    Part.from_text(base_instructions),
    Part.from_text(
        """The following example illustrate how to apply the scene transition instructions above. Pay attention to how the scene transitions are identified and described based on the provided video and VTT data."""
    ),
    Part.from_uri(one_shot_video_path, "video/mp4"),
    Part.from_uri(one_shot_vtt_path, "text/vtt"),
    Part.from_text(
        """Scene transition of the earlier shared video and VTT file: Scene1: Timecode: start_time 00:00:06 – end_time 00:01:00 - description Carroll Shelby and Henry Ford II discuss the car's top speed in a garage \n Scene2: Timecode: start_time 00:01:00 – end_time 00:01:54 - description  Henry Ford II is helped into the car on the tarmac. \n Scene3: Timecode: start_time 00:01:55 – end_time 00:03:45 - description The dialogue transitions from technical aspects to the thrill of the driving experience, and the setting changes from static indoors to dynamic action on the tarmac. so on.."""
    ),
]

In [ ]:
responses = model.generate_content(oneshot_parts, generation_config=config)

In [ ]:
model_outputs = []

for candidate in responses.candidates:
    for part in candidate.content.parts:
        if part.text:
            try:
                parsed = json.loads(part.text)
                model_outputs.append(parsed)
            except json.JSONDecodeError as e:
                print("JSON parsing failed:", e)

In [ ]:
def candidate_count_handler_with_tolerance(responses, tolerance_seconds=1):
    """
    Consolidates scene transition predictions from multiple candidates (e.g., LLM responses),
    merging timestamps within a given tolerance and selecting the most common reason.

    Args:
        responses: An object with a `candidates` attribute, where each candidate contains a
                   `.text` field that is a JSON string of scene predictions.
                   Each prediction is expected to be a list of dicts with keys:
                   - "scene_number" (int)
                   - "timestamp" (str, format: "HH:MM:SS")
                   - "reason" (str)
        tolerance_seconds (int): Allowed deviation (in seconds) between timestamps to consider them duplicates.

    Returns:
        List[Dict]: A consolidated list of scene transitions, each with:
                    - "scene_number": int
                    - "timestamp": str ("HH:MM:SS")
                    - "reason": str (most common across candidates)
    """
    candidates = [json.loads(candidate.text) for candidate in responses.candidates]
    scene_data = defaultdict(lambda: {"timestamps": [], "reasons": Counter()})

    for candidate in candidates:
        for scene in candidate:
            scene_number = scene["scene_number"]
            timestamp = datetime.strptime(scene["timestamp"], "%H:%M:%S")
            reason = scene["reason"]
            scene_data[scene_number]["timestamps"].append(timestamp)
            scene_data[scene_number]["reasons"][reason] += 1

    final_result = []
    for scene_number, data in scene_data.items():
        timestamps = sorted(data["timestamps"])
        consolidated_timestamp = timestamps[0]
        for ts in timestamps:
            if abs((ts - consolidated_timestamp).total_seconds()) > tolerance_seconds:
                consolidated_timestamp = ts

        most_common_reason = data["reasons"].most_common(1)[0][0]
        final_result.append(
            {
                "scene_number": scene_number,
                "timestamp": consolidated_timestamp.strftime("%H:%M:%S"),
                "reason": most_common_reason,
            }
        )

    return final_result

In [ ]:
model_output = candidate_count_handler_with_tolerance(responses, tolerance_seconds=1)
model_output